# 01 E-Feature Extraction

Build an `EModelEFeatureExtractionScanConfig`, generate `TaskConfig` entities,
and run the extraction either **locally** or on **Fargate**.

This notebook demonstrates:
1. Discovering protocols and amplitudes from recordings
2. Building the extraction configuration
3. Inspecting the generated `TaskConfig` and inputs to BluePyEfe
4. Running the extraction locally (or launching on Fargate)

**Reads from:** entitycore staging (`ElectricalCellRecording` entities).

**Writes to:** `obi-output/01_efeature_extraction/grid_scan/0/` containing:
- `obi_one_coordinate.json` — the TaskConfig
- `ephys_data/<entity-id>/` — downloaded NWB files
- `config/recipes.json` and `config/extract_config/targets.json`
- `figures/` — extraction plots
- `extracted_features.json` — the output for optimisation

## Imports

In [ ]:
import json
from pathlib import Path

import obi_one as obi
from entitysdk import Client, ProjectContext
from obi_auth import get_token
from obi_one.core.info import Info
from obi_one.scientific.tasks.emodel_building.task1_efeature_extraction.blocks.initialize import (
    ExtractionInitialize,
)
from obi_one.scientific.tasks.emodel_building.task1_efeature_extraction.blocks.settings import (
    Settings,
)

## Connect to entitycore staging

In [ ]:
virtual_lab_id = obi.LAB_ID_STAGING_TEST
project_id = obi.PROJECT_ID_STAGING_TEST

token = get_token(environment="staging")
project_context = ProjectContext(virtual_lab_id=virtual_lab_id, project_id=project_id)
db_client = Client(
    api_url="https://staging.openbraininstitute.org/api/entitycore",
    project_context=project_context,
    token_manager=token,
)
print("Connected to entitycore staging.")

## Select recordings and discover protocols

The `/declared/mapped-electrical-cell-recording-properties` endpoint reads the
protocol names from each recording's entity metadata and downloads the NWB files
to discover per-protocol step amplitudes (nA).

In [ ]:
import requests

OBI_ONE_URL = "http://127.0.0.1:8100"

# ElectricalCellRecording entities from the staging test project
RECORDING_IDS = (
    "00854004-4390-4a42-bf9d-5e672e8c8484",
    "66fd126c-f478-47e9-88ad-d12f7f27c84f",
    "0b952fa9-efad-44a2-bdf9-fef732af6702",
    # "d1362042-dd22-4945-9bb7-bdc3c0f25a8f",
    # "6d0945a3-325d-4da9-9099-8ee65fac6a4c"
)

# Discover protocols and amplitudes from the recordings
url = f"{OBI_ONE_URL}/declared/mapped-electrical-cell-recording-properties"
params = {"recording_ids": list(RECORDING_IDS)}
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/json",
}
if virtual_lab_id:
    headers["virtual-lab-id"] = virtual_lab_id
if project_id:
    headers["project-id"] = project_id

response = requests.get(url, headers=headers, params=params, timeout=120)
response.raise_for_status()
data = response.json()

protocol_union = data["Protocols"]
by_recording = data["ProtocolsByRecording"]
amplitudes_by_protocol = data["AmplitudesByProtocol"]

print(f"Discovered {len(protocol_union)} protocols: {protocol_union}")
print()
print("Protocols per recording:")
for rid, protocols in by_recording.items():
    print(f"  {rid[:8]}...: {protocols}")
print()
print("Step amplitudes (nA) per protocol:")
for proto, amps in amplitudes_by_protocol.items():
    print(f"  {proto}: {len(amps)} amplitudes")

## Build the scan config from discovered protocols

Instead of using hardcoded default protocols, we build Protocol instances only
for protocols that actually exist in the recordings. Each protocol is populated
with its discovered amplitudes.

In [ ]:
from obi_one.scientific.tasks.emodel_building.task1_efeature_extraction.protocols_and_features import (
    protocols as protocol_module,
)

# Map class names to Protocol classes
PROTOCOL_CLASSES = {
    cls.__name__: cls
    for cls in [
        protocol_module.IDRestProtocol,
        protocol_module.IDThreshProtocol,
        protocol_module.IVProtocol,
        protocol_module.APWaveformProtocol,
        protocol_module.SAHPProtocol,
        protocol_module.GenericStepProtocol,
        protocol_module.SpikeRecProtocol,
        protocol_module.RampProtocol,
    ]
}

# Build protocols only for those discovered in recordings
VALIDATION_AMPLITUDES: set[tuple[str, float]] = set()  # e.g. {("IDRestProtocol", 0.25)}

protocols_list = []
for class_name in protocol_union:
    if class_name not in PROTOCOL_CLASSES:
        print(f"Warning: No Protocol class for {class_name}, skipping")
        continue
    
    protocol = PROTOCOL_CLASSES[class_name]()
    amps = amplitudes_by_protocol.get(class_name, [])
    protocol.extraction_amplitudes = tuple(
        (amp, (class_name, amp) in VALIDATION_AMPLITUDES) for amp in amps
    )
    protocols_list.append(protocol)

print(f"Built {len(protocols_list)} protocols with amplitudes:")
for p in protocols_list:
    n_val = sum(1 for _, is_val in p.extraction_amplitudes if is_val)
    print(
        f"  {type(p).__name__} (protocol_name='{p.protocol_name}'): "
        f"{len(p.extraction_amplitudes)} amplitudes ({n_val} validation), "
        f"{len(p.features)} features"
    )

In [ ]:
from obi_one.scientific.tasks.emodel_building.task1_efeature_extraction.blocks.protocol_and_feature_selection import (
    ProtocolAndFeatureSelection,
    SelectEFeaturesByProtocol,
)

scan_config = obi.EModelEFeatureExtractionScanConfig(
    info=Info(
        campaign_name="L5PC eFeature Extraction",
        campaign_description="Extract e-features from staging test recordings.",
    ),
    initialize=ExtractionInitialize(
        electrical_cell_recording=tuple(
            obi.ElectricalCellRecordingFromID(id_str=rid) for rid in RECORDING_IDS
        ),
    ),
    settings=Settings(),
    efeatures_by_protocol=ProtocolAndFeatureSelection(
        selection=SelectEFeaturesByProtocol(protocols=tuple(protocols_list))
    ),
)

print("Scan config created.")
print(f"  Recordings: {len(RECORDING_IDS)}")
print(f"  Protocols: {len(scan_config.efeatures_by_protocol.selection.protocols)}")
print(f"  Global eFEL settings: {scan_config.settings.global_efel_settings()}")

## Generate TaskConfig and inspect inputs

Run `GridScanGenerationTask` to generate the coordinate config files and register
`TaskConfig` entities in entitycore. This does NOT run the extraction yet.

In [ ]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../../../../obi-output/01_efeature_extraction/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute(db_client=db_client)

coord_root = Path(grid_scan.single_configs[0].coordinate_output_root).resolve()
print(f"Coordinate output root: {coord_root}")

## Inspect the generated TaskConfig

The `obi_one_coordinate.json` file contains the full configuration that will be
passed to `EModelEFeatureExtractionTask`. This is what BluePyEfe will receive.

In [ ]:
config_path = coord_root / "obi_one_coordinate.json"
task_config = json.loads(config_path.read_text())

print("=== TaskConfig Structure ===")
print(f"Top-level keys: {list(task_config.keys())}")
print()

# Show recordings
recordings = task_config.get("initialize", {}).get("electrical_cell_recording", [])
print(f"Recordings ({len(recordings)}):")
for r in recordings:
    print(f"  - {r.get('id_str', 'unknown')}")
print()

# Show settings
settings = task_config.get("settings", {})
print("Settings:")
print(f"  spike_detection_threshold: {settings.get('spike_detection_threshold')}")
print(f"  trace_resampling_timestep: {settings.get('trace_resampling_timestep')}")
print(f"  default_std_value: {settings.get('default_std_value')}")

In [ ]:
# Show protocols that will be sent to BluePyEfe
protocols = (
    task_config.get("efeatures_by_protocol", {})
    .get("selection", {})
    .get("protocols", [])
)

print(f"=== Protocols for BluePyEfe ({len(protocols)}) ===")
print()
for p in protocols:
    ptype = p.get("type", "unknown")
    # The 'type' field contains the class name; BluePyEfe uses protocol_name from the class
    amps = p.get("extraction_amplitudes", [])
    features = p.get("features", [])
    
    print(f"{ptype}:")
    print(f"  amplitudes: {len(amps)} entries")
    if amps:
        amp_values = [a[0] for a in amps[:5]]
        print(f"    first 5: {amp_values}")
    print(f"  features: {len(features)}")
    if features:
        feature_names = [f.get('type', 'unknown') for f in features[:5]]
        print(f"    first 5: {feature_names}")
    print()

## Registered TaskConfig entities

The `GridScanGenerationTask.execute()` registered `TaskConfig` entities in entitycore.

In [ ]:
campaign = grid_scan.form.campaign
campaign_id = campaign.id if campaign else None

single = grid_scan.single_configs[0].single_entity
single_id = single.id if single else None

print("=== Registered TaskConfig Entities ===")
print(f"  Campaign TaskConfig: {campaign_id}")
print(f"  Single TaskConfig:   {single_id}")

---

# Option A: Run extraction locally

Run the extraction task directly in this notebook. This downloads the NWB files,
runs BluePyEfe, and writes outputs to `coord_root`.

In [ ]:
from obi_one.scientific.tasks.emodel_building.task1_efeature_extraction.task import (
    EModelEFeatureExtractionTask,
)

# Get the single config (already has coordinate_output_root set)
single_config = grid_scan.single_configs[0]

# Create and execute the task
task = EModelEFeatureExtractionTask(config=single_config)
result_path = task.execute(db_client=db_client)

print(f"Extraction complete. Output: {result_path}")

## Inspect extraction outputs

In [ ]:
print(f"Output directory: {coord_root}")
print()

# List generated files
for item in sorted(coord_root.rglob("*")):
    if item.is_file():
        rel = item.relative_to(coord_root)
        size = item.stat().st_size
        print(f"  {rel} ({size:,} bytes)")

In [ ]:
# Inspect extracted features
features_path = coord_root / "extracted_features.json"
if features_path.exists():
    features = json.loads(features_path.read_text())
    print("=== Extracted Features ===")
    print(f"Top-level keys: {list(features.keys())}")
    print(f"Number of efeatures: {len(features.get('efeatures', []))}")
    print(f"Number of protocols: {len(features.get('protocols', []))}")
else:
    print(f"File not found: {features_path}")
    print("Run the local extraction cell above first.")

In [ ]:
# Inspect the recipes.json (what was sent to BluePyEModel)
recipes_path = coord_root / "config" / "recipes.json"
if recipes_path.exists():
    recipes = json.loads(recipes_path.read_text())
    ps = recipes["emodel"]["pipeline_settings"]
    print("=== BluePyEModel Recipe ===")
    print(f"validation_protocols: {ps.get('validation_protocols')}")
    print(f"extract_absolute_amplitudes: {ps.get('extract_absolute_amplitudes')}")
    print(f"efel_settings: {ps.get('efel_settings')}")
    print(f"minimum_protocol_delay: {ps.get('minimum_protocol_delay')}")
else:
    print(f"File not found: {recipes_path}")

In [ ]:
# Inspect targets.json (what BluePyEfe extracts)
targets_path = coord_root / "config" / "extract_config" / "targets.json"
if targets_path.exists():
    targets = json.loads(targets_path.read_text())
    print("=== BluePyEfe Targets ===")
    print(f"Number of files: {len(targets.get('files', []))}")
    print(f"Number of targets: {len(targets.get('targets', []))}")
    print()
    # Show first few targets
    for t in targets.get('targets', [])[:5]:
        print(f"  {t.get('protocol')} @ {t.get('amplitude')} nA: {t.get('efeature')}")
    if len(targets.get('targets', [])) > 5:
        print(f"  ... and {len(targets.get('targets', [])) - 5} more")
else:
    print(f"File not found: {targets_path}")

---

# Option B: Launch on Fargate

Instead of running locally, submit the task to run on Fargate via the obi-one
service. This is useful for large extractions or when you want to run remotely.

In [ ]:
import requests

url = f"{OBI_ONE_URL}/declared/task/launch"
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}
if virtual_lab_id:
    headers["virtual-lab-id"] = virtual_lab_id
if project_id:
    headers["project-id"] = project_id

response = requests.post(
    url,
    json={"task_type": "efeature_extraction", "config_id": str(single_id)},
    headers=headers,
    timeout=30,
)
if not response.ok:
    print(f"Error {response.status_code}: {response.text}")
    print()
    print("Check that the obi-one server is running and reachable.")
else:
    launch_info = response.json()
    print("Job launched successfully!")
    print(f"  activity_id: {launch_info['activity_id']}")
    print(f"  job_id:      {launch_info['job_id']}")

In [ ]:
# After the Fargate job completes, query the TaskActivity for generated_ids:
# (Uncomment and run after the job finishes)

# from entitysdk.models import TaskActivity
# activity = db_client.get_entity(
#     entity_id=launch_info["activity_id"],
#     entity_type=TaskActivity,
# )
# print(f"Generated IDs: {activity.generated_ids}")
# print()
# print("Copy the TaskResult ID into notebook 02:")
# print(f'  TaskResultFromID(id_str="{activity.generated_ids[0]}")')